# NL2BI — Colab Text-to-SQL `/extract` service

End-to-end runner: mount Drive → load env → fetch repo → install deps → load Qwen2.5-Coder-7B (4-bit) → boot FastAPI on `:8000` → expose via ngrok → smoke-test `/health` and `/extract`.

Run **Runtime → Run all** after attaching a GPU runtime (T4 / L4 / A100). Secrets (`NGROK_AUTHTOKEN`, `GITHUB_PAT`) live in `/content/drive/MyDrive/nl2bi_colab/.env`; never paste them into a cell.

In [3]:
# 1. Mount Google Drive (persists logs/artifacts and holds .env with secrets).
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

from pathlib import Path
DRIVE_BASE = Path('/content/drive/MyDrive/nl2bi_colab')
for sub in ('logs', 'artifacts', 'repo'):
    (DRIVE_BASE / sub).mkdir(parents=True, exist_ok=True)
print('Drive base:', DRIVE_BASE)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive base: /content/drive/MyDrive/nl2bi_colab


In [4]:
# 2. Load env from Drive `.env` (NGROK_AUTHTOKEN, GITHUB_PAT, etc.) and set defaults.
import os
from pathlib import Path

ENV_PATH = Path('/content/drive/MyDrive/nl2bi_colab/.env')
loaded = 0
if ENV_PATH.exists():
    for line in ENV_PATH.read_text(encoding='utf-8').splitlines():
        line = line.strip()
        if not line or line.startswith('#') or '=' not in line:
            continue
        key, value = line.split('=', 1)
        key, value = key.strip(), value.strip().strip('"').strip("'")
        os.environ[key] = value  # always overwrite, .env on Drive is the source of truth
        loaded += 1
    print(f'Loaded {loaded} keys from {ENV_PATH}')
else:
    print(f'No .env at {ENV_PATH} — falling back to existing env')

os.environ.setdefault('COLAB_MODEL_ID', 'Qwen/Qwen2.5-Coder-7B-Instruct')
os.environ.setdefault('COLAB_QUANTIZATION', '4bit')
os.environ.setdefault('COLAB_MAX_NEW_TOKENS', '512')
os.environ.setdefault('COLAB_MOCK_MODEL', 'false')
os.environ.setdefault('COLAB_PORT', '8000')
os.environ.setdefault('COLAB_SPIDER_DB_ROOT', '/content/drive/MyDrive/diploma_plan_sql/data/spider/database')
os.environ.setdefault('COLAB_DATA_SOURCES_PATH', '/content/nl2bi-colab/demo_data/data_sources.json')
os.environ.setdefault('COLAB_DEFAULT_DATA_SOURCE_ID', 'demo_sales')
os.environ.setdefault('COLAB_ARTIFACTS_DIR', '/content/drive/MyDrive/nl2bi_colab/artifacts')
os.environ.setdefault('COLAB_LOG_DIR', '/content/drive/MyDrive/nl2bi_colab/logs')

print('NGROK_AUTHTOKEN present:', bool(os.environ.get('NGROK_AUTHTOKEN')), '(value hidden)')
print('GITHUB_PAT present:', bool(os.environ.get('GITHUB_PAT')), '(value hidden)')

Loaded 12 keys from /content/drive/MyDrive/nl2bi_colab/.env
NGROK_AUTHTOKEN present: True (value hidden)
GITHUB_PAT present: True (value hidden)


In [5]:
# 3. Clone (or update) the repo. Uses GIT_HTTPS_TOKEN from .env for the private repo.
import os, subprocess, shutil
from pathlib import Path
from urllib.parse import quote

GIT_OWNER = os.environ.get('NL2BI_GIT_OWNER', 'petrussia')
GIT_REPO = os.environ.get('NL2BI_GIT_REPO', 'NL2BI-AI-assistant')
GIT_BRANCH = os.environ.get('NL2BI_GIT_BRANCH', 'integration/colab-text-to-sql')
REPO_DIR = Path('/content/nl2bi-colab')
TOKEN = os.environ.get('GITHUB_PAT')

if TOKEN:
    GIT_URL = f'https://x-access-token:{quote(TOKEN, safe="")}@github.com/{GIT_OWNER}/{GIT_REPO}.git'
else:
    GIT_URL = f'https://github.com/{GIT_OWNER}/{GIT_REPO}.git'

def _run(cmd, **kwargs):
    res = subprocess.run(cmd, capture_output=True, text=True, **kwargs)
    if res.returncode != 0:
        # Strip the token from any error output before printing.
        out = (res.stdout + res.stderr).replace(TOKEN or 'NO_TOKEN', '<token>')
        raise RuntimeError(f'`{" ".join(cmd)}` failed (rc={res.returncode}):\n{out}')
    return res

is_repo = (REPO_DIR / '.git').exists()
if REPO_DIR.exists() and not is_repo:
    # Stale partial dir from a previous failed clone: wipe it.
    print('Wiping stale', REPO_DIR)
    shutil.rmtree(REPO_DIR)
    is_repo = False

if is_repo:
    _run(['git', '-C', str(REPO_DIR), 'remote', 'set-url', 'origin', GIT_URL])
    _run(['git', '-C', str(REPO_DIR), 'fetch', '--all', '--prune'])
    _run(['git', '-C', str(REPO_DIR), 'checkout', GIT_BRANCH])
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], capture_output=True, text=True)
else:
    if not TOKEN:
        raise RuntimeError(
            'GITHUB_PAT missing. Add it to /content/drive/MyDrive/nl2bi_colab/.env\n'
            '  GITHUB_PAT=ghp_xxx_with_repo_read_scope\n'
            'then re-run cells 2 and 3.'
        )
    _run(['git', 'clone', '--branch', GIT_BRANCH, GIT_URL, str(REPO_DIR)])

# Make sure the token never lingers in the on-disk remote URL.
_run(['git', '-C', str(REPO_DIR), 'remote', 'set-url', 'origin',
      f'https://github.com/{GIT_OWNER}/{GIT_REPO}.git'])

print('Repo at:', REPO_DIR)
print('Tree top:', sorted(p.name for p in REPO_DIR.iterdir() if not p.name.startswith('.')))
head = subprocess.run(['git', '-C', str(REPO_DIR), 'log', '-1', '--oneline'], capture_output=True, text=True).stdout.strip()
print('HEAD:', head)

Repo at: /content/nl2bi-colab
Tree top: ['README.md', 'colab', 'contracts', 'demo_data', 'docs']
HEAD: 9b52349 fix(colab): authenticate git clone for private repo + diagnose-friendly error output


In [6]:
# 4. Install pinned deps.
import subprocess, sys
REQ = '/content/nl2bi-colab/colab/requirements-colab.txt'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', REQ], check=True)
print('deps ok')

deps ok


In [7]:
# 5. GPU sanity check.
import sys
if '/content/nl2bi-colab' not in sys.path:
    sys.path.insert(0, '/content/nl2bi-colab')
from colab.gpu import gpu_info
import json
print(json.dumps(gpu_info(), ensure_ascii=False, indent=2))

{
  "device": "cuda",
  "gpu_name": "NVIDIA L4",
  "vram_total_gb": 22.03,
  "vram_free_gb": 21.84,
  "cuda_available": true
}


In [ ]:
# 6. Launch uvicorn in background. Idempotent: kills any prior server on the port first.
import os, subprocess, time
from pathlib import Path
from urllib import request, error
import json as _json

REPO_DIR = '/content/nl2bi-colab'
PORT = int(os.environ.get('COLAB_PORT', '8000'))
LOG_DIR = Path(os.environ['COLAB_LOG_DIR'])
LOG_DIR.mkdir(parents=True, exist_ok=True)
STDOUT_PATH = LOG_DIR / 'uvicorn.stdout.log'
STDERR_PATH = LOG_DIR / 'uvicorn.stderr.log'

def _env_true(name):
    return os.environ.get(name, '').strip().lower() in {'1', 'true', 'yes', 'y', 'on'}

def _tail(path, lines=80):
    try:
        text = Path(path).read_text(encoding='utf-8', errors='replace')
    except FileNotFoundError:
        return f'<missing: {path}>'
    return '\n'.join(text.splitlines()[-lines:])

def _kill_port(port):
    """Kill any process bound to the given TCP port (so re-running this cell works)."""
    killed = False
    try:
        out = subprocess.run(['fuser', '-k', f'{port}/tcp'], capture_output=True, text=True)
        killed = (out.returncode == 0)
    except FileNotFoundError:
        out = subprocess.run(
            ['bash', '-c', f"lsof -ti tcp:{port} | xargs -r kill -9"],
            capture_output=True, text=True,
        )
        killed = bool(out.stdout.strip())
    if killed:
        print(f'killed prior process on :{port}')
        time.sleep(1)

_kill_port(PORT)

if not _env_true('COLAB_MOCK_MODEL'):
    try:
        import torch
        cuda_ok = bool(torch.cuda.is_available())
    except Exception:
        cuda_ok = False
    if not cuda_ok:
        raise RuntimeError(
            'No CUDA GPU detected, but COLAB_MOCK_MODEL is false. '
            'Attach a GPU runtime (Runtime -> Change runtime type -> GPU) '
            'or set os.environ["COLAB_MOCK_MODEL"] = "true" before launching.'
        )

STDOUT = STDOUT_PATH.open('ab', buffering=0)
STDERR = STDERR_PATH.open('ab', buffering=0)
for handle, name in ((STDOUT, 'stdout'), (STDERR, 'stderr')):
    handle.write(f'\n--- launch {time.strftime("%Y-%m-%d %H:%M:%S")} {name} ---\n'.encode())

env = os.environ.copy()
env['PYTHONPATH'] = REPO_DIR + (':' + env['PYTHONPATH'] if env.get('PYTHONPATH') else '')

proc = subprocess.Popen(
    [
        'python', '-m', 'uvicorn',
        'colab.text_to_sql_colab_server:app',
        '--host', '0.0.0.0',
        '--port', str(PORT),
        '--log-level', 'info',
    ],
    cwd=REPO_DIR,
    env=env,
    stdout=STDOUT, stderr=STDERR,
)
print('uvicorn pid:', proc.pid)
print('stdout log:', STDOUT_PATH)
print('stderr log:', STDERR_PATH)

URL = f'http://127.0.0.1:{PORT}/health'
deadline = time.time() + 600
last_err = None
last_print = 0
while time.time() < deadline:
    rc = proc.poll()
    if rc is not None:
        raise RuntimeError(
            f'uvicorn exited early with code {rc}.\n\n'
            f'--- stderr tail ---\n{_tail(STDERR_PATH)}\n\n'
            f'--- stdout tail ---\n{_tail(STDOUT_PATH)}'
        )
    try:
        with request.urlopen(URL, timeout=5) as resp:
            payload = _json.loads(resp.read())
            if payload.get('model_loaded') or payload.get('mock_model'):
                print('health ok:')
                print(_json.dumps(payload, ensure_ascii=False, indent=2))
                break
            if time.time() - last_print >= 15:
                print('waiting for model_loaded ... status:')
                print(_json.dumps(payload, ensure_ascii=False, indent=2))
                last_print = time.time()
    except (error.URLError, TimeoutError, ConnectionError) as exc:
        last_err = exc
        if time.time() - last_print >= 15:
            print('waiting for /health ... last error:', repr(exc))
            print('stderr tail:')
            print(_tail(STDERR_PATH, lines=20))
            last_print = time.time()
    time.sleep(5)
else:
    raise RuntimeError(
        f'service did not become healthy in time: {last_err}\n\n'
        f'--- stderr tail ---\n{_tail(STDERR_PATH)}\n\n'
        f'--- stdout tail ---\n{_tail(STDOUT_PATH)}'
    )


In [9]:
# 7. Open ngrok tunnel. Token comes from env, never printed.
import os
from pyngrok import ngrok, conf

tok = os.environ.get('NGROK_AUTHTOKEN')
if not tok:
    raise RuntimeError('NGROK_AUTHTOKEN missing. Add it to /content/drive/MyDrive/nl2bi_colab/.env')
conf.get_default().auth_token = tok

for tunnel in list(ngrok.get_tunnels()):
    ngrok.disconnect(tunnel.public_url)

PORT = int(os.environ.get('COLAB_PORT', '8000'))
tunnel = ngrok.connect(PORT, 'http')
PUBLIC_URL = tunnel.public_url.replace('http://', 'https://')
print('PUBLIC_URL:', PUBLIC_URL)

PUBLIC_URL: https://b4c6-34-143-155-54.ngrok-free.app                                               


In [ ]:
# 8. Smoke-test against the public tunnel. Captures and prints stdout so output survives in the cell.
import subprocess, sys
res = subprocess.run(
    [sys.executable, '-m', 'colab.smoke_extract', '--base-url', PUBLIC_URL],
    cwd='/content/nl2bi-colab',
    capture_output=True, text=True,
)
print(res.stdout)
if res.stderr:
    print('--- stderr ---')
    print(res.stderr)
print(f'(smoke exit code: {res.returncode})')


In [11]:
# 9. (Optional) Stop server + tunnel before disconnecting runtime.
# from pyngrok import ngrok
# for t in list(ngrok.get_tunnels()):
#     ngrok.disconnect(t.public_url)
# proc.terminate(); proc.wait(timeout=10)
# print('stopped')
pass